# HadGEM3-GC31-MM

Has an issue with dim sizes...

```
2026-04-15 08:59:24 - INFO - failed for HadGEM3-GC31-MM with reason:
2026-04-15 08:59:24 - INFO - conflicting sizes for dimension 'lat': length 127 on 'ucomp_EOFs_s_DJF' and length 126 on {'time': 'time', 'pfull': 'pfull', 'lat': 'lat', 'time_DJF': 'time_DJF', 'time_JFM': 'time_JFM', 'time_FMA': 'time_FMA', 'time_MAM': 'time_MAM', 'time_AMJ': 'time_AMJ', 'time_MJJ': 'time_MJJ', 'time_JJA': 'time_JJA', 'time_JAS': 'time_JAS', 'time_ASO': 'time_ASO', 'time_SON': 'time_SON', 'time_OND': 'time_OND', 'time_NDJ': 'time_NDJ', 'eof_num': 'eof_num'}
2026-04-15 08:59:24 - INFO - continuing
```

In [1]:
import xarray as xr
import os
import numpy as np
from eofs.standard import Eof
import logging

from functions.SIT_functions.SIT_eddy_feedback_functions import eof_calc, vert_integrate, read_daily_averages, propagate_missing_data_to_all_vars

/home/users/cturrell/miniforge3/envs/eddy/lib/python3.10/site-packages/esmpy/interface/loadESMF.py:94: VersionWarning: ESMF installation version 8.8.0, ESMPy version 8.8.0b0
  warnings.warn("ESMF installation version {}, ESMPy version {}".format(


In [2]:
exp_type = 'cmip'
output_eof_file = '/home/users/cturrell/documents/eddy_feedback/b-parameter/cmip6_b/hist_NaN_issues/data'
force_eof_recalculate = True
pfull_slice = slice(850., 100.)
subtract_annual_cycle=True
eof_vars = ['ucomp', 'div1_QG', 'div1_QG_123', 'div1_QG_gt3']
n_eofs = 3
season_month_dict = {'DJF':[12,1,2,]}
propogate_all_nans = True

In [3]:
had_path = '/gws/ssde/j25a/arctic_connect/cturrell/CMIP6/historical/HadGEM3-GC31-MM/1850_2015/6hrPlevPt/'
anom_ds_had = xr.open_dataset(os.path.join(had_path, '1979_2015', 'anoms.nc'))
dataset_had = read_daily_averages(os.path.join(had_path, 'yearly_data'), start_month=1979, end_month=2015, exp_type='cmip6')

eof_had = eof_calc('cmip', output_eof_file, force_eof_recalculate, dataset_had, pfull_slice,
                              subtract_annual_cycle, eof_vars, n_eofs, season_month_dict, anom_ds_had,
                              propogate_all_nans, level_subset=[250., 500., 850.],
                              pressure_weighted=True)

2026-04-17 10:01:02,839 [INFO]: SIT_eddy_feedback_functions.py(read_daily_averages:1145) >> reading daily average files
2026-04-17 10:01:02,839 [INFO]: SIT_eddy_feedback_functions.py(read_daily_averages:1145) >> reading daily average files


2026-04-17 10:01:05,174 [INFO]: SIT_eddy_feedback_functions.py(read_daily_averages:1186) >> finished reading daily average files
2026-04-17 10:01:05,174 [INFO]: SIT_eddy_feedback_functions.py(read_daily_averages:1186) >> finished reading daily average files
2026-04-17 10:01:05,199 [INFO]: SIT_eddy_feedback_functions.py(check_for_duplicate_times:1211) >> duplicate times are present!!!
2026-04-17 10:01:05,199 [INFO]: SIT_eddy_feedback_functions.py(check_for_duplicate_times:1211) >> duplicate times are present!!!
2026-04-17 10:01:05,513 [INFO]: SIT_eddy_feedback_functions.py(check_for_duplicate_times:1215) >> duplicate times removed. Was 12996 now 12961
2026-04-17 10:01:05,513 [INFO]: SIT_eddy_feedback_functions.py(check_for_duplicate_times:1215) >> duplicate times removed. Was 12996 now 12961
2026-04-17 10:01:05,539 [INFO]: SIT_eddy_feedback_functions.py(check_for_duplicate_times:1218) >> duplicate times are NOT present - continuing
2026-04-17 10:01:05,539 [INFO]: SIT_eddy_feedback_funct

In [4]:
def obtain_orig_var_hem(exp_type, output_eof_file, force_eof_recalculate, dataset, pfull_slice,
             eof_vars, n_eofs, season_month_dict, anom_ds,
             propagate_all_nans, level_subset=None):  

    pfull_selector = level_subset if level_subset is not None else pfull_slice

    # Resolve level_subset to nearest actual levels in the data to avoid
    # exact-match failures on models with slightly different pressure coordinates
    if isinstance(pfull_selector, list):
        pfull_selector = [float(anom_ds['ucomp_anom'].sel(pfull=lev, method='nearest').pfull.values)
                          for lev in pfull_selector]

    if np.all(dataset.lat.diff('lat')<0.):
        hemisphere_slice_dict = {'n':slice(80.,10.),
                                's':slice(-10., -80.),                          
                                }
    else:
        hemisphere_slice_dict = {'n':slice(10.,80.),
                                's':slice(-80., -10.),                             
                                }        

    if propagate_all_nans:
        output_eof_file_use = output_eof_file.split('.nc')[0]+'_prop_nans.nc'
    else:
        output_eof_file_use = output_eof_file

    if os.path.isfile(output_eof_file_use) and not force_eof_recalculate:
        logging.info('attempting to read in EOF data')
        eof_ds = xr.open_mfdataset(output_eof_file_use, decode_times=True)
        logging.info('SUCCESS')
    else:
        logging.info('failed to read in previously calculated EOF data - CALCULATING')

        logging.info('calculating anomalies')
        if exp_type=='held_suarez':
            u_anoms = dataset['ucomp'].mean('lon') - dataset['ucomp'].mean(('time','lon'))
        else:
            # u_anoms_time = dataset.temporal.departures(data_var = 'ucomp', freq='day', weighted=False)['ucomp'].mean('lon')
            u_anoms_time = anom_ds['ucomp_anom']

        season_list = [season_val for season_val in season_month_dict.keys()]

        all_time_season_list = season_list+['all_time']

        eof_ds = xr.Dataset()
        eof_ds.coords['time'] = ('time', u_anoms_time.time.values)
        eof_ds.coords['pfull'] = ('pfull', u_anoms_time.sel(pfull=pfull_selector).pfull.values)  
        eof_ds.coords['lat'] = ('lat', u_anoms_time.sel(lat=hemisphere_slice_dict['n']).lat.values)                

        for season_val in season_list:
            u_anoms_season = u_anoms_time.where(anom_ds['time'].dt.month.isin(season_month_dict[season_val]), drop=True)
            eof_ds.coords[f'time_{season_val}'] = (f'time_{season_val}', u_anoms_season.time.values)

        eof_ds.coords['eof_num'] = ('eof_num', np.arange(n_eofs))

        ucomp_solver_dict = {}
        ucomp_500_solver_dict = {}        
        ucomp_va_solver_dict = {}        
        for season_val in all_time_season_list:
            ucomp_solver_dict[season_val] = {'n':{}, 's':{}}
            ucomp_va_solver_dict[season_val] = {'n':{}, 's':{}}
            ucomp_500_solver_dict[season_val] = {'n':{}, 's':{}}

        logging.info('about to propagate nans to all fields')
        if propagate_all_nans:
            anom_ds = propagate_missing_data_to_all_vars(anom_ds, eof_vars)

        for eof_var in eof_vars:

            logging.info('loading anomalies from anom_ds')

            var_anoms = anom_ds[f'{eof_var}_anom'].load()
            orig_var = anom_ds[f'{eof_var}_orig'].load()           

            for hemisphere in ['n','s']:

                for time_frame in all_time_season_list:
                    var_anoms_hem = var_anoms.sel(lat=hemisphere_slice_dict[hemisphere]).sel(pfull=pfull_selector)
                    
                    if time_frame!='all_time':
                        var_anoms_hem = var_anoms_hem.where(eof_ds['time'].dt.month.isin(season_month_dict[time_frame]), drop=True) 
                        time_dim_name = f'time_{time_frame}'
                    else:
                        time_dim_name = 'time'

                    orig_var_hem = orig_var.sel(lat=hemisphere_slice_dict[hemisphere]).sel(pfull=pfull_selector)
                    
                    if time_frame!='all_time':
                        orig_var_hem = orig_var_hem.where(eof_ds['time'].dt.month.isin(season_month_dict[time_frame]), drop=True)
                        
                    orig_var_hem = orig_var_hem.mean('time')   
                    
                    va_var_anoms_hem = vert_integrate(var_anoms_hem, pressure_weighted=True)

    return var_anoms_hem, va_var_anoms_hem, anom_ds

orig, va, anom_ds = obtain_orig_var_hem(exp_type, output_eof_file, force_eof_recalculate, dataset_had, pfull_slice,
             eof_vars, n_eofs, season_month_dict, anom_ds_had, propagate_all_nans=True, level_subset=[250.,500.,850.])
anom_ds

2026-04-17 10:02:20,891 [INFO]: 3947695604.py(obtain_orig_var_hem:32) >> failed to read in previously calculated EOF data - CALCULATING
2026-04-17 10:02:20,891 [INFO]: 3947695604.py(obtain_orig_var_hem:32) >> failed to read in previously calculated EOF data - CALCULATING
2026-04-17 10:02:20,892 [INFO]: 3947695604.py(obtain_orig_var_hem:34) >> calculating anomalies
2026-04-17 10:02:20,892 [INFO]: 3947695604.py(obtain_orig_var_hem:34) >> calculating anomalies


2026-04-17 10:02:20,988 [INFO]: 3947695604.py(obtain_orig_var_hem:64) >> about to propagate nans to all fields
2026-04-17 10:02:20,988 [INFO]: 3947695604.py(obtain_orig_var_hem:64) >> about to propagate nans to all fields
2026-04-17 10:02:23,227 [INFO]: SIT_eddy_feedback_functions.py(propagate_missing_data_to_all_vars:660) >> successfully propagated missing data from all vars to all other vars to enable eof projection, so each field now has 9110237 nans
2026-04-17 10:02:23,227 [INFO]: SIT_eddy_feedback_functions.py(propagate_missing_data_to_all_vars:660) >> successfully propagated missing data from all vars to all other vars to enable eof projection, so each field now has 9110237 nans
2026-04-17 10:02:23,229 [INFO]: 3947695604.py(obtain_orig_var_hem:70) >> loading anomalies from anom_ds
2026-04-17 10:02:23,229 [INFO]: 3947695604.py(obtain_orig_var_hem:70) >> loading anomalies from anom_ds
2026-04-17 10:02:23,445 [INFO]: 3947695604.py(obtain_orig_var_hem:70) >> loading anomalies from an

<xarray.Dataset> Size: 2GB
Dimensions:           (pfull: 6, lat: 325, time: 12961)
Coordinates:
  * pfull             (pfull) float64 48B 925.0 850.0 700.0 600.0 500.0 250.0
  * lat               (lat) float64 3kB -90.0 -89.44 -88.89 ... 88.89 89.44 90.0
  * time              (time) object 104kB 1979-01-01 00:00:00 ... 2015-01-01 ...
Data variables:
    ucomp_anom        (time, pfull, lat) float64 202MB nan nan ... -0.371 0.0
    ucomp_orig        (time, pfull, lat) float32 101MB nan nan ... 0.2145 0.0
    div1_QG_anom      (time, pfull, lat) float64 202MB nan nan ... -1.166e+08
    div1_QG_orig      (time, pfull, lat) float64 202MB nan nan ... 0.1104 1.707
    div1_QG_123_anom  (time, pfull, lat) float64 202MB nan nan ... -0.9716 2.147
    div1_QG_123_orig  (time, pfull, lat) float64 202MB nan nan ... 0.1142 3.102
    div1_QG_gt3_anom  (time, pfull, lat) float64 202MB nan nan ... -1.166e+08
    div1_QG_gt3_orig  (time, pfull, lat) float64 202MB nan nan ... -1.395

In [5]:
print(anom_ds.ucomp_anom.size)
print(anom_ds.ucomp_anom.isnull().sum())

print((anom_ds.ucomp_anom.isnull().sum()/anom_ds.ucomp_anom.size)*100)

25273950
<xarray.DataArray 'ucomp_anom' ()> Size: 8B
array(9110237)
<xarray.DataArray 'ucomp_anom' ()> Size: 8B
array(36.04595641)


In [6]:
subset = anom_ds.ucomp_anom.sel(lat=slice(10., 80.)).sel(pfull=[250., 500., 850.], method='nearest')

# How many spatial points are contaminated (have ANY NaN across time)?
n_contaminated = subset.isnull().any('time').sum().values
n_total_spatial = subset.sizes['lat'] * subset.sizes['pfull']
print(f"Contaminated spatial points: {n_contaminated} / {n_total_spatial}")

# The killer question:
print(f"ALL spatial points have ≥1 NaN: {subset.isnull().any('time').all().values}")


Contaminated spatial points: 129 / 378
ALL spatial points have ≥1 NaN: False


In [7]:
def eof_calc_alt(data,lats):

    coslat = np.cos(np.deg2rad(lats)).clip(0., 1.)
    wgts = np.sqrt(coslat)[np.newaxis, np.newaxis, :]

    solver = Eof(data, weights=wgts, center=True)

    eofs = solver.eofsAsCovariance(neofs=3)
    pc1 = solver.pcs(npcs=3, pcscaling=1)

    variance_fractions = solver.varianceFraction(neigs=3)

    return eofs, pc1, variance_fractions, solver

eofs_va, pc1_va, variance_fractions_va, solver_va = eof_calc_alt(anom_ds.ucomp_anom.values, anom_ds.ucomp_anom.lat.values)

In [8]:
anom_ds.ucomp_anom

<xarray.DataArray 'ucomp_anom' (time: 12961, pfull: 6, lat: 325)> Size: 202MB
array([[[        nan,         nan,         nan, ..., -3.20905158,
         -1.66521697,  0.        ],
        [        nan,         nan,         nan, ..., -3.21775224,
         -1.56529823,  0.        ],
        [        nan,         nan,         nan, ..., -1.4238101 ,
         -0.91506702,  0.        ],
        [ 0.        , -2.0629402 , -2.85643256, ..., -2.17018368,
         -1.11162386,  0.        ],
        [ 0.        , -1.26440623, -2.31272431, ..., -3.04187162,
         -1.56723055,  0.        ],
        [ 0.        , -0.96420785, -2.15564365, ..., -1.34153736,
         -0.69640401,  0.        ]],

       [[        nan,         nan,         nan, ...,  3.88593307,
          2.28949121,  0.        ],
        [        nan,         nan,         nan, ...,  3.04793728,
          1.68456167,  0.        ],
        [        nan,         nan,         nan, ...,  1.83692948,
          1.0380248 ,  0.        ],
        [ 0.        , -0.51417643, -1.6574858 , ...,  1.01506086,
...
         -0.19389977,  0.        ],
        [ 0.        , -0.04674015, -0.02133115, ..., -0.22241911,
         -0.28753421,  0.        ],
        [ 0.        ,  0.36449706,  0.52066061, ..., -0.88688736,
         -0.5223883 ,  0.        ],
        [ 0.        ,  0.17470914,  0.36114341, ..., -1.90626891,
         -1.04124517,  0.        ]],

       [[        nan,         nan,         nan, ..., -1.65184578,
         -1.40025323,  0.        ],
        [        nan,         nan,         nan, ..., -1.5610384 ,
         -0.36017541,  0.        ],
        [        nan,         nan,         nan, ..., -2.6400756 ,
         -1.71812886,  0.        ],
        [ 0.        , -0.07571636, -0.16596455, ..., -3.01757111,
         -1.70360318,  0.        ],
        [ 0.        , -0.44499215, -0.38699783, ..., -2.61062832,
         -1.01573303,  0.        ],
        [ 0.        ,  0.37162451,  0.79817503, ..., -1.31701522,
         -0.37102559,  0.        ]]])
Coordinates:
  * pfull    (pfull) float64 48B 925.0 850.0 700.0 600.0 500.0 250.0
  * lat      (lat) float64 3kB -90.0 -89.44 -88.89 -88.33 ... 88.89 89.44 90.0
  * time     (time) object 104kB 1979-01-01 00:00:00 ... 2015-01-01 00:00:00
Attributes:
    operation:   temporal_avg
    mode:        departures
    freq:        day
    weighted:    True
    min_weight:  0.0